# Whisper — fix and sweep

**Established so far.** `large-v3-turbo` returned `en` on 75 of 99 clips and
translated rather than transcribed; delegating language ID to `large-v3` fixes
that. Both variants then produce roughly **half** the reference words, with ~50%
of the error being deletions. Beam size and `condition_on_previous_text` were
A/B tested across all four combinations and change nothing. At least part of it
is repetition collapse — one clip transcribes 47 s correctly then repeats a
single five-word phrase for the remaining 550 s.

**Sarvam reaches ratio 0.96 on this same audio**, so the reference word counts
are right and the audio is transcribable. The difference is that Sarvam runs
**per-segment**: handed one diarized turn at a time, it must return something
for each, while long-form Whisper decides for itself what to skip.

This notebook tests that directly, then sweeps if it holds. Turns come from the
**fusion**, which is also what the Sarvam sweep now uses — so the two systems
differ only in the recogniser.

In [2]:
# --- sync + config ----------------------------------------------------------
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/sarvam-assignment")
if (CODE/".git").exists():
    subprocess.run(["git","-C",str(CODE),"fetch","-q","origin","main"], check=True)
    subprocess.run(["git","-C",str(CODE),"reset","-q","--hard","origin/main"], check=True)
else:
    subprocess.run(["git","clone","-q",
                    "https://github.com/ParvGoyal08/MultilingualASR.git", str(CODE)], check=True)
print("code @", subprocess.run(["git","-C",str(CODE),"log","-1","--format=%h %s"],
                               capture_output=True, text=True).stdout.strip())
for m in [k for k in list(sys.modules) if k=="sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[m]
sys.path.insert(0, str(CODE))

from sarvam_diar.config import Config, StageFlags
from sarvam_diar import asr, data, diarization, reference, refinement, text_metrics as tm
import torch, dataclasses, collections

# Kaggle sessions do NOT share /kaggle/working -- it is per-session and wiped
# on close. So this notebook cannot rely on main_kaggle having populated
# anything: it relinks the audio dataset itself, and restores the diarization
# hypotheses from a previous run attached as an Input.
from sarvam_diar import utils
ROOT = Path("/kaggle/working/sarvam_diarization")
ROOT.mkdir(parents=True, exist_ok=True)
print("dataset:", utils.relink_dataset(ROOT, "/kaggle/input", dataset=None))

# Diarization comes from an attached dataset. Found by CONTENT, not by slug --
# Kaggle slugs move, and nesting varies (/kaggle/input/<slug>/ for an uploaded
# dataset, /kaggle/input/datasets/<user>/<slug>/ for some attachments). This is
# the same reasoning as relink_dataset, which finds the audio by looking for
# audio_16k/ rather than by a hard-coded path.
import shutil as _sh
PREV_RESULTS = None      # set only to pin one dataset and skip the search

_found = ([Path(PREV_RESULTS)] if PREV_RESULTS else
          sorted({q.parent for d in (1, 2, 3, 4)
                  for q in Path("/kaggle/input").glob("/".join(["*"]*d) + "/hypotheses")}))
for _prev in _found:
    for _sub in ("hypotheses", "results", "reference"):
        _src = next((c for c in (_prev/"sarvam_diarization"/_sub, _prev/_sub)
                     if c.is_dir()), None)
        if _src:
            _sh.copytree(_src, ROOT/_sub, dirs_exist_ok=True)
    _models = sorted(q.name for q in (ROOT/"hypotheses").glob("*") if q.is_dir())
    print(f"restored from {_prev.name}: {_models}")
if not _found:
    # Kaggle usually unpacks an uploaded zip into the dataset, but not always.
    # If nothing exposes a hypotheses/ directory, look inside any attached zip
    # for one rather than failing with "attach a dataset" when they already did.
    import zipfile
    for _z in Path("/kaggle/input").glob("**/*.zip"):
        with zipfile.ZipFile(_z) as _zf:
            if any(n.startswith("hypotheses/") for n in _zf.namelist()):
                _zf.extractall(ROOT)
                print(f"extracted {_z.name} ->",
                      sorted(q.name for q in (ROOT/"hypotheses").glob("*") if q.is_dir()))
                _found = [_z]
                break
if not _found:
    print("no attached dataset contains hypotheses/ -- see the check at the "
          "bottom of this cell for what to do")

cfg = Config.create(root=ROOT, work_dir=ROOT/"tmp")
CLIPS = {c.clip_id: c for c in data.parse_ground_truth(data.load_segments_csv(cfg))}
INPUTS, _ = data.split_reference(list(CLIPS.values()), None, cfg=cfg)
REFS = {cid: reference.build_reference(CLIPS[cid]) for cid in INPUTS}
READY = [dataclasses.replace(ci, wav_path=str(cfg.wav_path(cid)))
         for cid, ci in INPUTS.items() if cfg.wav_path(cid).exists()]

MODEL = "large-v3-turbo"
SYSTEM = f"whisper-{MODEL}"
FUSION_MODELS = ["community-1", "reverb-v2", "diarizen-large"]
DIAR = "fusion"

# Report every device. Both Whisper models default to cuda:0, so on a 2xT4
# session ~5 GB of weights plus both activation peaks land on one card and it
# runs out of memory while the other sits idle. asr._lid_device() now puts
# language ID on the second card when there is one.
if torch.cuda.is_available():
    for _d in range(torch.cuda.device_count()):
        _free, _tot = torch.cuda.mem_get_info(_d)
        print(f"cuda:{_d} {torch.cuda.get_device_name(_d)}  "
              f"{_free/2**30:.1f} GB free of {_tot/2**30:.1f} GB")
    print("language ID will use cuda:", asr._lid_device())
else:
    print("no GPU")
print("clips with audio:", len(READY))

# The bench writes no asr/ checkpoint -- that is main_kaggle's job -- but it
# does need fusion TURNS to cut on. A prebuilt fusion is enough; the three
# source diarizations are only required if one has to be built here.
# Only clips that HAVE audio can have turns. One of the 100 failed
# extraction in Step 1, so checking all of REFS reports a permanent
# shortfall of 1 and makes a complete fusion look broken.
_have_audio = {c.clip_id for c in READY}
_need = [c for c in _have_audio if not diarization.is_done(cfg, DIAR, c)]
if _need:
    _src = [m for m in FUSION_MODELS
            if not all(diarization.is_done(cfg, m, c) for c in _have_audio)]
    if _src:
        _avail = sorted(str(q.relative_to("/kaggle/input"))
                        for q in Path("/kaggle/input").glob("*/*")) or ["(nothing)"]
        raise RuntimeError(
            f"{len(_need)} clips have no {DIAR} turns, and the sources to build "
            f"them ({_src}) are not here either.\n"
            f"  Attach a dataset containing hypotheses/fusion/ and point "
            f"PREV_RESULTS at it.\n"
            f"  The repo packages one at local_out/upload/fusion_rttm.zip "
            f"(150 KB, 99 clips).\n"
            f"  currently attached: {_avail}")
    # deterministic, skips finished clips, atomic -- safe to re-run
    refinement.materialise(cfg, REFS, FUSION_MODELS, threshold=0.5)
print(f"fusion clips: "
      f"{sum(1 for c in _have_audio if diarization.is_done(cfg, DIAR, c))}"
      f"/{len(_have_audio)} (of clips with audio)")


code @ fbc9719 IndicConformer: never let pip touch torch


ImportError: text_metrics needs rapidfuzz for word alignment: pip install rapidfuzz

In [ ]:
_done = [c for c in REFS if diarization.is_done(cfg, DIAR, c)]
print(f"fusion clips: {len(_done)}/{len(REFS)}")
for c in [c for c in REFS if c not in _done][:3]:
    print(" ", c, "rttm:", cfg.hyp_rttm_path(DIAR, c).exists(),
          "sidecar:", cfg.hyp_meta_path(DIAR, c).exists())

fusion clips: 99/100
  GUVrL5ltiP4__23_86 rttm: False sidecar: False


## 1 — long-form vs per-segment

In [ ]:
# --- the decisive test: does per-segment fix the deletions? -----------------
# Long-form Whisper decides for itself what to transcribe and drops roughly half
# the words. Sarvam reaches ratio 0.96 on the same audio running PER SEGMENT,
# which removes that choice: it is handed one turn at a time and must return
# something for each. If that is the difference, Whisper per-segment should also
# land near 1.0 -- and it is the strategy that makes the two systems comparable.
import time, gc

def _free_gpu():
    """Drop cached blocks between configurations.

    Three configurations run back to back in one kernel, and CTranslate2 holds
    its workspace until the model object is released. Without this the peak is
    the sum of the configurations rather than the largest of them.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


probe = sorted(READY, key=lambda c: c.duration)[:3]
print(f"{'strategy':<34}{'ref w':>7}{'hyp w':>7}{'ratio':>7}{'WER':>9}{'del%':>7}{'sec':>6}")

def score(pairs_by_clip):
    rw = hw = d = s_ = i_ = h = 0
    for cid, hyp in pairs_by_clip.items():
        ref = REFS[cid]
        r = [t for u in ref.utterances for t in reference.tokenize(u.text_norm)]
        c = tm.wer_counts(r, hyp)
        rw += len(r); hw += len(hyp)
        d += c.deletions; s_ += c.substitutions; i_ += c.insertions; h += c.hits
    n = h + s_ + d
    return rw, hw, hw/max(rw,1), (s_+d+i_)/max(n,1), d/max(n,1)

# long-form, current settings
t0 = time.time(); out = {}
for c in probe:
    words, _ = asr.transcribe_whisper(cfg, Path(c.wav_path), word_timestamps=False,
                                      beam_size=5, lid_model=asr.LID_MODEL)
    out[c.clip_id] = [t for w in words
                      for t in reference.normalize_text(w.text, strip_gloss=False).split()]
print(f"{'long-form (current)':<34}" + "{:>7}{:>7}{:>7.2f}{:>9.4f}{:>7.1%}".format(*score(out))
      + f"{time.time()-t0:>6.0f}")

# long-form with the discard threshold off
t0 = time.time(); out = {}
for c in probe:
    words, _ = asr.transcribe_whisper(cfg, Path(c.wav_path), word_timestamps=False,
                                      beam_size=5, lid_model=asr.LID_MODEL,
                                      no_speech_threshold=None)
    out[c.clip_id] = [t for w in words
                      for t in reference.normalize_text(w.text, strip_gloss=False).split()]
_free_gpu()
print(f"{'long-form, no_speech off':<34}" + "{:>7}{:>7}{:>7.2f}{:>9.4f}{:>7.1%}".format(*score(out))
      + f"{time.time()-t0:>6.0f}")

# per-segment over the fusion, exactly how Sarvam is run
t0 = time.time(); out = {}
for c in probe:
    turns = asr.merge_same_speaker(diarization.load_hypothesis(cfg, DIAR, c.clip_id), 1.0)
    segs, _m = asr.transcribe_segments(cfg, SYSTEM, Path(c.wav_path), turns)
    out[c.clip_id] = [t for sg in segs
                      for t in reference.normalize_text(sg["text"], strip_gloss=False).split()]
_free_gpu()
print(f"{'PER-SEGMENT on fusion':<34}" + "{:>7}{:>7}{:>7.2f}{:>9.4f}{:>7.1%}".format(*score(out))
      + f"{time.time()-t0:>6.0f}")

print("\nratio near 1.0 = producing about as many words as were spoken.")
print("Sarvam per-segment on this corpus: 0.96.")


strategy                            ref w  hyp w  ratio      WER   del%   sec
17:19:57 | INFO    | sarvam_diar | loading faster-whisper large-v3 on cuda:0 (float16) -- the first call also downloads the weights


RuntimeError: CUDA failed with error out of memory

## 1b — configuration probe: 9 configurations x 10 clips

The three-clip test above compares *strategies*. This compares *configurations*, on ten clips covering all nine scripts plus one long clip, and reports the two things WER hides: whether the output is in the right **script** at all, and how much of it is a single repeated phrase.

Writes no `asr/` checkpoint — it calls `transcribe_whisper` / `transcribe_segments` directly, so it cannot collide with a sweep running elsewhere. Row **A** costs no GPU: it scores the existing checkpoints on the same ten clips, making the comparison paired.

In [ ]:
# --- five configurations, ten clips, no checkpoints written -----------------
# Logic lives in tools/whisper_probe.py rather than in this cell, because the
# Kaggle notebook is its own copy: library code arrives with the pull in cell 1,
# an edited cell does not.
import importlib, json as _json
sys.path.insert(0, str(CODE/"tools"))
import whisper_probe; importlib.reload(whisper_probe)

IDS = whisper_probe.select_clips(cfg, 10)
_miss = [c for c in IDS if not diarization.is_done(cfg, DIAR, c)]
assert not _miss, f"no {DIAR} turns for {_miss}"
print(f"{len(IDS)} clips, "
      f"{sum(CLIPS[c].end_sec - CLIPS[c].start_sec for c in IDS)/60:.1f} min of audio")
for c in IDS:
    print(f"   {c:<28}{CLIPS[c].stats.get('lang_script','?'):<12}"
          f"{CLIPS[c].end_sec - CLIPS[c].start_sec:>6.0f}s")

# A  old greedy/self-LID   B beam5+LID   C  B + cond_prev=False
# D  per-segment (beam 1)  E large-v3+own LID
# F/G/H/I are the ORACLE-LANGUAGE twins of B/C/E/D -- same decoding,
# language taken from the reference, so each pair prices language ID.
# ONLY = {"F","G","H","I"} runs just the new ones; None runs all nine.
ONLY = {"F", "G", "H", "I"}
RUN_PROBE = True

if RUN_PROBE:
    _res = whisper_probe.probe(cfg, IDS, diar=DIAR, model=MODEL, only=ONLY)
    _out = Path(cfg.root)/"results"/"whisper_probe10.json"
    _out.parent.mkdir(parents=True, exist_ok=True)
    _out.write_text(_json.dumps(_res, indent=2, default=str))
    print(f"\nwrote {_out}")
    print("script = clips whose output is in the reference's script "
          "(a good ratio in English prose is still a failure)")
    print("rep5   = share of output covered by its most frequent 5-gram; "
          "median across the corpus is 3.0%, so >20% is collapse")
else:
    print("RUN_PROBE is False")


16:51:06 | INFO    | sarvam_diar | segments CSV already cached (/kaggle/working/sarvam_diarization/data/youtube_segments.csv)
16:51:07 | INFO    | sarvam_diar | parsed 100 clips, 9940 segments, 2 dropped as malformed, 0 unparsable entries
10 clips, 19.4 min of audio
   GfFLEIAAumk__90_163         Bengali         73s
   T3I2T-cfhG4__160_210        Devanagari      50s
   L7xRazDdtgw__19_83          Gujarati        64s
   8r2Nltl0W4o__259_320        Gurmukhi        61s
   PRAzUz0GANs__223_283        Kannada         60s
   7L4gi7Ncc0s__90_148         Malayalam       58s
   7YfsQPYY-W0__351_411        Oriya           60s
   HZv_WvIr6lE__21_105         Tamil           84s
   ARZl7LT0UC0__73_130         Telugu          57s
   0AEEA8NyVwY__11_609         Devanagari     598s
16:51:07 | INFO    | sarvam_diar | segments CSV already cached (/kaggle/working/sarvam_diarization/data/youtube_segments.csv)
16:51:07 | INFO    | sarvam_diar | parsed 100 clips, 9940 segments, 2 dropped as malformed, 0 unp

## 1c — IndicConformer-600M: smoke test

Verifies download and access, the device it actually runs on, 16 kHz mono tensor input, and one RNNT transcription against the reference — before anything is run at scale.

`onnxruntime-gpu` is installed **here** rather than in cell 1 because it can conflict with `ctranslate2` (faster-whisper). If it breaks the environment, only this notebook is affected: restart and re-run cell 1.

The model loads with `trust_remote_code=True`, i.e. it executes code fetched from the Hub. Standard for this model, stated because it is worth knowing.

In [ ]:
# --- IndicConformer preflight + smoke ---------------------------------------
# DO NOT follow the model card's install line here. It says
#   pip install transformers torchaudio onnxruntime-gpu
# and on Kaggle torch and torchaudio are preinstalled against the image's CUDA
# build. Letting pip resolve them upgrades torch underneath the running kernel,
# and the failure surfaces far away as
#   AttributeError: module 'torch' has no attribute '_utils'
# from inside an unrelated transformers import. A kernel restart does NOT fix
# it, because the replaced files persist for the session -- only a Factory
# reset does. So: report first, install only what is genuinely absent, and
# never anything that can pull torch.
import importlib
sys.path.insert(0, str(CODE/"tools"))
import whisper_probe as wp; importlib.reload(wp)
from sarvam_diar import asr_indic; importlib.reload(asr_indic)

_state = asr_indic.preflight()
for _k, _v in _state.items():
    print(f"  {_k:<14}{_v}")

_need = asr_indic.install_hint(_state)
_broken = [k for k, v in _state.items() if isinstance(v, str) and v.startswith("BROKEN")]
if _broken:
    raise RuntimeError(
        f"{_broken} are broken in this session. Session options > Factory reset, "
        "then re-run cell 1 and this cell. Do not pip install torch or torchaudio.")

INSTALL = True
if INSTALL and _need:
    print("\ninstalling (no torch, no torchaudio):", _need)
    _r = _sp.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *_need],
                 capture_output=True, text=True)
    print("pip rc", _r.returncode, (_r.stderr or "").strip()[-300:])
    print("\n*** RESTART THE KERNEL, then re-run cell 1 and this cell ***")
elif not _need:
    print("\nnothing to install -- everything needed is already present")


AttributeError: module 'sarvam_diar.asr_indic' has no attribute 'preflight'

In [ ]:
# --- the smoke test itself (run after any install + restart) ----------------
IDS = wp.select_clips(cfg, 10)
_cid = IDS[0]
_lang = wp.oracle_language(REFS[_cid], CLIPS[_cid].stats)
print(f"smoke clip {_cid}  language {_lang} (from the reference)\n")

_s = asr_indic.smoke(cfg, _cid, _lang, decoding="rnnt", seconds=20.0)
for _k, _v in _s.items():
    print(f"  {_k:<14}{str(_v)[:300]}")

_gt = " ".join(u.text_norm for u in REFS[_cid].utterances)
print(f"\n  GT (first 300 chars)\n  {_gt[:300]}")
print("\nPASS if: device is cuda, wav_shape is (1, N), and the text is in the "
      "reference's script rather than empty or Latin.")


## 1d — IndicConformer on the same 10 clips

Per-segment on the fusion — IndicConformer is an utterance-level Conformer, so long-form is not an option and segments are capped at 30 s. Same clips, same scoring and same table as §1b, so the rows are directly comparable to Whisper's.

**Language is the oracle**, taken from the reference. IndicConformer has no language identification of its own, so some source is mandatory; this row is therefore an ablation, and its leak-free counterpart is `lang_source="lid"`.

In [ ]:
# --- IndicConformer, RNNT and CTC, oracle language, per-segment on fusion ---
import indic_probe; importlib.reload(indic_probe)

RUN_INDIC = True
if RUN_INDIC:
    _res = indic_probe.probe(cfg, IDS, diar=DIAR, lang_source="oracle",
                             decodings=("rnnt", "ctc"))
    _out = Path(cfg.root)/"results"/"indic_probe10.json"
    _out.parent.mkdir(parents=True, exist_ok=True)
    _out.write_text(_json.dumps(_res, indent=2, default=str))
    print(f"\nwrote {_out}")
else:
    print("RUN_INDIC is False")


## 2 — verdict (the sweep itself belongs in `main_kaggle`)

In [ ]:
# This notebook DECIDES; it does not sweep. Everything above runs in memory over
# a handful of clips and writes no checkpoint, so nothing here can collide with
# a run in main_kaggle or leave a half-written result someone later scores.
#
# If the test above shows per-segment reaching a ratio near 1.0, the finding is
# promoted rather than executed here: set RUN_WHISPER = True in main_kaggle 3.4,
# which owns every asr/ checkpoint in the project.
print("verdict -> set RUN_WHISPER in main_kaggle 3.4 if per-segment reached ~1.0")


verdict -> set RUN_WHISPER in main_kaggle 3.4 if per-segment reached ~1.0


## 3 — score

In [ ]:
# --- score whatever is on disk ---------------------------------------------
import json as _json
_norm = lambda t: reference.normalize_text(t, strip_gloss=False)

def _pairs(system, cid):
    p = _json.loads(asr.asr_path(cfg, system, cid).read_text())
    if p.get("strategy") == "segment":
        return [(t, s["speaker"]) for s in sorted(p["segments"], key=lambda s: s["start"])
                for t in _norm(s["text"]).split()]
    turns = diarization.load_hypothesis(cfg, DIAR, cid)
    return [(t, spk) for w, spk in asr.assign_words(asr.load_words(cfg, system, cid), turns)
            for t in _norm(w).split()]

_found = sorted(d.name for d in (cfg.root/"asr").iterdir()) if (cfg.root/"asr").exists() else []
print(f"{'system':<34}{'clips':>6}{'ratio':>7}{'WER':>9}{'cpWER':>9}{'WDER':>8}")
for _s in _found:
    rows = []
    for cid, ref in REFS.items():
        if not asr.is_done(cfg, _s, cid):
            continue
        if "@" not in _s and not diarization.is_done(cfg, DIAR, cid):
            continue
        pr = _pairs(_s, cid)
        r = tm.score_transcript(
            {k: v.split() for k, v in reference.speaker_texts(ref).items()},
            asr.speaker_texts_from_words(pr), reference.word_stream(ref), pr)
        r["n_hyp"] = len(pr); rows.append(r)
    if not rows:
        continue
    g = tm.summarise(rows)
    ratio = sum(r["n_hyp"] for r in rows)/max(g["n_ref_words"],1)
    print(f"{_s:<34}{g['n_clips']:>6}{ratio:>7.2f}{g['wer']:>9.4f}"
          f"{g['cpwer']:>9.4f}{g['wder']:>8.4f}")


system                             clips  ratio      WER    cpWER    WDER
sarvam-saaras-v3@fusion               97   1.01   0.3059   0.3373  0.0814
sarvam-saaras-v3@reverb-v2            99   0.96   0.2728   0.3957  0.1128
sarvam-saaras-v4@reverb-v2             9   0.96   0.2606   0.3485  0.0718
whisper-large-v3                      35   0.31   0.9296   0.9340  0.2664
whisper-large-v3-turbo                99   0.75   0.9827   0.9957  0.5471
